# Latencia end-to-end — Todas las arquitecturas (H4)

Mide latencia por pregunta para 6 arquitecturas sobre N=5 preguntas del test set.

**Orden de ejecución:**
1. Celda 1 — instalar deps → **Restart session**
2. Celda 2 — instalar Ollama
3. Celda 3 — montar Drive + configurar rutas
4. Celdas 4 en adelante — ejecutar en orden

**Subir al panel de archivos (Files):** `dataset_test.json`

**Requisitos Drive (`TFM_RGPD/`):**
- `db_rgpd_pymupdf/`
- `db_rgpd_semantic/`
- `colab_checkpoints/` (checkpoints LoRA del fine-tuning)

**Output:** `TFM_RGPD/latency_results/` (6 JSON + 1 resumen CSV)

In [ ]:
# CELDA 1 — Instalar dependencias (restart session despues)
!pip install -q langchain langchain-community langchain-ollama transformers peft bitsandbytes accelerate \
    sentence-transformers rank-bm25 chromadb huggingface_hub

In [ ]:
# CELDA 2 — Instalar Ollama + pull llama3:8b
import subprocess, time, os

subprocess.run('apt-get install -y zstd', shell=True, capture_output=True)
result = subprocess.run('curl -fsSL https://ollama.ai/install.sh | sh', shell=True, capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('Fallo instalando Ollama: ' + result.stderr[-300:])

which = subprocess.run('which ollama', shell=True, capture_output=True, text=True)
OLLAMA = which.stdout.strip() or '/usr/local/bin/ollama'
assert os.path.exists(OLLAMA), f'No encontrado: {OLLAMA}'

subprocess.Popen([OLLAMA, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
subprocess.run([OLLAMA, 'pull', 'llama3:8b'])
print('Ollama listo:', subprocess.run([OLLAMA, 'list'], capture_output=True, text=True).stdout)

In [ ]:
# CELDA 3 — Drive + HF login + rutas
import os, json, re, time
from google.colab import drive
drive.mount('/content/drive')

# ── Ajusta si tus carpetas tienen otro nombre ─────────────────────────────────
TFM_DRIVE       = '/content/drive/MyDrive/TFM_RGPD'
DB_PYMUPDF      = f'{TFM_DRIVE}/db_rgpd_pymupdf'
DB_SEMANTIC     = f'{TFM_DRIVE}/db_rgpd_semantic'
CHECKPOINTS_DIR = f'{TFM_DRIVE}/colab_checkpoints'
OUTPUT_DIR      = f'{TFM_DRIVE}/latency_results'
DATASET_PATH    = '/content/dataset_test.json'
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
for d in [DB_PYMUPDF, DB_SEMANTIC, CHECKPOINTS_DIR]:
    if not os.path.exists(d):
        raise FileNotFoundError(f'No encontrado en Drive: {d}')
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError('Sube dataset_test.json al panel de archivos de Colab')

from huggingface_hub import login
HF_TOKEN = 'hf_XXXXXXXXXXXXXXXXXXXXXXXX'  # reemplaza con tu token
login(token=HF_TOKEN)
print('Drive y HF OK')

In [ ]:
# CELDA 4 — Parsear dataset + constantes compartidas
N_SAMPLES = 5
MODEL_ID  = 'meta-llama/Meta-Llama-3-8B-Instruct'

def parse_dataset(path, n=N_SAMPLES):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            obj = json.loads(line)
            text = obj.get('text', '')
            ctx_m  = re.search(r'Contexto:\s*(.*?)\n\nPregunta:', text, re.DOTALL)
            preg_m = re.search(r'Pregunta:\s*(.*?)<\|eot_id\|>', text)
            gt_m   = re.search(r'assistant<\|end_header_id\|>\n(.*?)(?:<\|eot_id\|>|$)', text, re.DOTALL)
            if preg_m and gt_m:
                data.append({
                    'question':     preg_m.group(1).strip(),
                    'ground_truth': gt_m.group(1).strip(),
                    'context':      ctx_m.group(1).strip() if ctx_m else '',
                })
    return data[:n]

dataset = parse_dataset(DATASET_PATH)
print(f'Dataset: {len(dataset)} preguntas')

TEMPLATE_RAG = """Eres un consultor juridico especializado en el RGPD. Responde de forma DIRECTA y CONCISA.
REGLAS:
- Usa UNICAMENTE la informacion del contexto proporcionado.
- Si el articulo exacto aparece en el contexto, citalo.
- Si la respuesta no esta en el contexto, responde: 'El contexto no contiene informacion suficiente.'
- Responde siempre en espanol.

Contexto legal del RGPD:
{context}

Pregunta: {question}

Respuesta directa (maximo 3-4 frases):"""

all_results = {}  # acumula todas las latencias
print('Constantes listas')

## 1. RAG Simple — ChromaDB pymupdf + Ollama

In [ ]:
# CELDA 5 — RAG Simple latencia
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_ollama import OllamaLLM

print('Cargando embeddings bge-m3 (GPU)...')
emb_rag = HuggingFaceEmbeddings(model_name='BAAI/bge-m3', model_kwargs={'device': 'cuda'})
vdb_rag = Chroma(persist_directory=DB_PYMUPDF, embedding_function=emb_rag)
ret_rag = vdb_rag.as_retriever(search_kwargs={'k': 5})
llm_rag = OllamaLLM(model='llama3:8b', temperature=0.0)

results_rag = []
print(f'RAG Simple ({N_SAMPLES} preguntas)...')
for i, e in enumerate(dataset):
    t0 = time.perf_counter()
    docs = ret_rag.invoke(e['question'])
    ctx = '\n\n'.join(d.page_content for d in docs)
    resp = llm_rag.invoke(TEMPLATE_RAG.format(context=ctx, question=e['question']))
    lat = round(time.perf_counter() - t0, 3)
    results_rag.append({'question': e['question'], 'answer': resp, 'contexts': [d.page_content for d in docs], 'ground_truth': e['ground_truth'], 'latency_s': lat})
    print(f'  [{i+1}/{N_SAMPLES}] {lat}s')

lats = [r['latency_s'] for r in results_rag]
print(f'\nRAG Simple => media: {sum(lats)/len(lats):.2f}s | min: {min(lats):.2f}s | max: {max(lats):.2f}s')
with open(f'{OUTPUT_DIR}/rag_latency_results.json', 'w', encoding='utf-8') as f:
    json.dump(results_rag, f, ensure_ascii=False, indent=4)
all_results['RAG Simple'] = lats

del vdb_rag, ret_rag

## 2. RAG Avanzado — BM25 + ChromaDB semantic + CrossEncoder + HyDE + Ollama

In [ ]:
# CELDA 6 — RAG Avanzado latencia
import torch
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from langchain_ollama import OllamaLLM

print('Cargando embeddings bge-m3 (GPU)...')
emb_adv = HuggingFaceEmbeddings(model_name='BAAI/bge-m3', model_kwargs={'device': 'cuda'})
vdb_adv = Chroma(persist_directory=DB_SEMANTIC, embedding_function=emb_adv)

print('Preparando BM25...')
raw = vdb_adv.get()
bm25_docs = [Document(page_content=t, metadata=m) for t, m in zip(raw['documents'], raw['metadatas'])]
bm25_adv = BM25Retriever.from_documents(bm25_docs, k=10)

print('Cargando CrossEncoder BAAI/bge-reranker-v2-m3 (CPU)...')
ce_tok = AutoTokenizer.from_pretrained('BAAI/bge-reranker-v2-m3')
ce_mod = AutoModelForSequenceClassification.from_pretrained('BAAI/bge-reranker-v2-m3')
ce_mod.eval()

K_RRF, TOP_N = 60, 5

def rerank(query, docs):
    pairs_q = [query] * len(docs)
    pairs_d = [d.page_content for d in docs]
    with torch.no_grad():
        enc = ce_tok(pairs_q, pairs_d, padding=True, truncation=True, max_length=512, return_tensors='pt')
        logits = ce_mod(**enc).logits.squeeze(-1)
        if logits.dim() == 0: logits = logits.unsqueeze(0)
        scores = logits.tolist()
    return [d for _, d in sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)[:TOP_N]]

def retrieve_adv(query):
    b = bm25_adv.invoke(query)
    c = vdb_adv.similarity_search(query, k=10)
    sc, dm = {}, {}
    for rk, d in enumerate(b):
        k = d.page_content[:200]; sc[k] = sc.get(k, 0.0) + 0.4 / (K_RRF + rk + 1); dm[k] = d
    for rk, d in enumerate(c):
        k = d.page_content[:200]; sc[k] = sc.get(k, 0.0) + 0.6 / (K_RRF + rk + 1); dm[k] = d
    fused = [dm[k] for k in sorted(sc, key=sc.__getitem__, reverse=True)[:10]]
    return rerank(query, fused)

llm_adv = OllamaLLM(model='llama3:8b', temperature=0.0)

def hyde(q):
    return llm_adv.invoke(
        'Genera un parrafo breve de documentacion legal del RGPD que responderia '
        'directamente a esta pregunta. Solo el fragmento, sin preambulo.\n\n'
        f'Pregunta: {q}\nFragmento hipotetico:'
    ).strip()

results_rag_adv = []
print(f'RAG Avanzado ({N_SAMPLES} preguntas)...')
for i, e in enumerate(dataset):
    t0 = time.perf_counter()
    q_exp = hyde(e['question'])
    docs = retrieve_adv(q_exp)
    ctx = '\n\n'.join(d.page_content for d in docs)
    resp = llm_adv.invoke(TEMPLATE_RAG.format(context=ctx, question=e['question']))
    lat = round(time.perf_counter() - t0, 3)
    results_rag_adv.append({'question': e['question'], 'answer': resp, 'contexts': [d.page_content for d in docs], 'ground_truth': e['ground_truth'], 'latency_s': lat})
    print(f'  [{i+1}/{N_SAMPLES}] {lat}s')

lats = [r['latency_s'] for r in results_rag_adv]
print(f'\nRAG Avanzado => media: {sum(lats)/len(lats):.2f}s | min: {min(lats):.2f}s | max: {max(lats):.2f}s')
with open(f'{OUTPUT_DIR}/rag_advanced_latency_results.json', 'w', encoding='utf-8') as f:
    json.dump(results_rag_adv, f, ensure_ascii=False, indent=4)
all_results['RAG Avanzado'] = lats

import gc
del vdb_adv, bm25_adv, ce_mod, ce_tok, emb_adv
gc.collect(); torch.cuda.empty_cache()

## 3. Baseline — Llama-3-8B sin fine-tuning, sin RAG

In [ ]:
# CELDA 7 — Baseline latencia
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                              bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=False)

print('Cargando Llama-3-8B base 4-bit...')
tok_base = AutoTokenizer.from_pretrained(MODEL_ID)
tok_base.pad_token = tok_base.eos_token
mod_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_cfg, trust_remote_code=True)
mod_base.eval()
print('Modelo listo')

results_baseline = []
print(f'Baseline ({N_SAMPLES} preguntas)...')
for i, e in enumerate(dataset):
    prompt = (
        '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n'
        'Eres un asistente legal experto en el RGPD. Responde basandote UNICAMENTE en el contexto.'
        '<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n'
        f'Contexto: {e["context"]}\n\nPregunta: {e["question"]}'
        '<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
    )
    t0 = time.perf_counter()
    inputs = tok_base(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = mod_base.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True,
                                 repetition_penalty=1.2, eos_token_id=tok_base.eos_token_id)
    resp = tok_base.decode(out[0], skip_special_tokens=True).split('assistant')[-1].strip()
    lat = round(time.perf_counter() - t0, 3)
    results_baseline.append({'question': e['question'], 'answer': resp, 'contexts': [e['context']], 'ground_truth': e['ground_truth'], 'latency_s': lat})
    print(f'  [{i+1}/{N_SAMPLES}] {lat}s')

lats = [r['latency_s'] for r in results_baseline]
print(f'\nBaseline => media: {sum(lats)/len(lats):.2f}s | min: {min(lats):.2f}s | max: {max(lats):.2f}s')
with open(f'{OUTPUT_DIR}/baseline_latency_results.json', 'w', encoding='utf-8') as f:
    json.dump(results_baseline, f, ensure_ascii=False, indent=4)
all_results['Baseline'] = lats

del mod_base, tok_base
gc.collect(); torch.cuda.empty_cache()

## 4. Fine-tuning — Llama-3-8B + LoRA

In [ ]:
# CELDA 8 — Fine-tuning latencia
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from pathlib import Path

bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                              bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=False)

checkpoints = sorted(Path(CHECKPOINTS_DIR).glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[1]))
if not checkpoints:
    raise FileNotFoundError(f'No hay checkpoints en {CHECKPOINTS_DIR}')
adapter_path = str(checkpoints[-1])
print(f'Checkpoint: {adapter_path}')

print('Cargando Llama-3-8B base 4-bit...')
tok_ft = AutoTokenizer.from_pretrained(MODEL_ID)
tok_ft.pad_token = tok_ft.eos_token
mod_ft = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_cfg, trust_remote_code=True)
mod_ft = PeftModel.from_pretrained(mod_ft, adapter_path)
mod_ft.eval()
print('Modelo fine-tuned listo')

results_ft = []
print(f'Fine-tuning ({N_SAMPLES} preguntas)...')
for i, e in enumerate(dataset):
    prompt = (
        '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n'
        'Eres un asistente legal experto en el RGPD. Responde basandote UNICAMENTE en el contexto.'
        '<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n'
        f'Contexto: {e["context"]}\n\nPregunta: {e["question"]}'
        '<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
    )
    t0 = time.perf_counter()
    inputs = tok_ft(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = mod_ft.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True,
                               repetition_penalty=1.2, eos_token_id=tok_ft.eos_token_id)
    resp = tok_ft.decode(out[0], skip_special_tokens=True).split('assistant')[-1].strip()
    lat = round(time.perf_counter() - t0, 3)
    results_ft.append({'question': e['question'], 'answer': resp, 'contexts': [e['context']], 'ground_truth': e['ground_truth'], 'latency_s': lat})
    print(f'  [{i+1}/{N_SAMPLES}] {lat}s')

lats = [r['latency_s'] for r in results_ft]
print(f'\nFine-tuning => media: {sum(lats)/len(lats):.2f}s | min: {min(lats):.2f}s | max: {max(lats):.2f}s')
with open(f'{OUTPUT_DIR}/fine_tuning_latency_results.json', 'w', encoding='utf-8') as f:
    json.dump(results_ft, f, ensure_ascii=False, indent=4)
all_results['Fine-tuning'] = lats

del mod_ft, tok_ft
gc.collect(); torch.cuda.empty_cache()

## 5. Híbrido — ChromaDB pymupdf + Llama-3 fine-tuned

In [ ]:
# CELDA 9 — Híbrido latencia
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from pathlib import Path
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                              bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=False)

checkpoints = sorted(Path(CHECKPOINTS_DIR).glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[1]))
adapter_path = str(checkpoints[-1])

print('Cargando embeddings bge-m3 (GPU)...')
emb_hyb = HuggingFaceEmbeddings(model_name='BAAI/bge-m3', model_kwargs={'device': 'cuda'})
vdb_hyb = Chroma(persist_directory=DB_PYMUPDF, embedding_function=emb_hyb)
ret_hyb = vdb_hyb.as_retriever(search_kwargs={'k': 5})

print('Cargando generador fine-tuned 4-bit...')
tok_hyb = AutoTokenizer.from_pretrained(MODEL_ID)
tok_hyb.pad_token = tok_hyb.eos_token
mod_hyb = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_cfg, trust_remote_code=True)
mod_hyb = PeftModel.from_pretrained(mod_hyb, adapter_path)
mod_hyb.eval()
print('Listo')

results_hyb = []
print(f'Hibrido ({N_SAMPLES} preguntas)...')
for i, e in enumerate(dataset):
    t0 = time.perf_counter()
    docs = ret_hyb.invoke(e['question'])
    ctx = '\n\n'.join(d.page_content for d in docs)
    prompt = (
        '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n'
        'Eres un asistente legal experto en el RGPD. Responde basandote UNICAMENTE en el contexto.'
        '<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n'
        f'Contexto: {ctx}\n\nPregunta: {e["question"]}'
        '<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
    )
    inputs = tok_hyb(prompt, return_tensors='pt', truncation=True, max_length=4096).to('cuda')
    with torch.no_grad():
        out = mod_hyb.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True,
                                repetition_penalty=1.2, eos_token_id=tok_hyb.eos_token_id)
    inp_len = inputs['input_ids'].shape[1]
    resp = tok_hyb.decode(out[0][inp_len:], skip_special_tokens=True).strip()
    lat = round(time.perf_counter() - t0, 3)
    results_hyb.append({'question': e['question'], 'answer': resp, 'contexts': [d.page_content for d in docs], 'ground_truth': e['ground_truth'], 'latency_s': lat})
    print(f'  [{i+1}/{N_SAMPLES}] {lat}s')

lats = [r['latency_s'] for r in results_hyb]
print(f'\nHibrido => media: {sum(lats)/len(lats):.2f}s | min: {min(lats):.2f}s | max: {max(lats):.2f}s')
with open(f'{OUTPUT_DIR}/hybrid_latency_results.json', 'w', encoding='utf-8') as f:
    json.dump(results_hyb, f, ensure_ascii=False, indent=4)
all_results['Hibrido'] = lats

del mod_hyb, tok_hyb, vdb_hyb, ret_hyb, emb_hyb
gc.collect(); torch.cuda.empty_cache()

## 6. Híbrido Avanzado — BM25 + CrossEncoder + HyDE + Llama-3 fine-tuned

In [ ]:
# CELDA 10 — Hibrido Avanzado latencia
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoModelForSequenceClassification
from peft import PeftModel
from pathlib import Path
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_ollama import OllamaLLM

bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                              bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=False)

checkpoints = sorted(Path(CHECKPOINTS_DIR).glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[1]))
adapter_path = str(checkpoints[-1])

print('Cargando embeddings bge-m3 (GPU)...')
emb_ha = HuggingFaceEmbeddings(model_name='BAAI/bge-m3', model_kwargs={'device': 'cuda'})
vdb_ha = Chroma(persist_directory=DB_SEMANTIC, embedding_function=emb_ha)

print('Preparando BM25...')
raw_ha = vdb_ha.get()
bm25_ha = BM25Retriever.from_documents(
    [Document(page_content=t, metadata=m) for t, m in zip(raw_ha['documents'], raw_ha['metadatas'])], k=10)

print('Cargando CrossEncoder (CPU)...')
ce_tok_ha = AutoTokenizer.from_pretrained('BAAI/bge-reranker-v2-m3')
ce_mod_ha = AutoModelForSequenceClassification.from_pretrained('BAAI/bge-reranker-v2-m3')
ce_mod_ha.eval()

def rerank_ha(q, docs):
    with torch.no_grad():
        enc = ce_tok_ha([q]*len(docs), [d.page_content for d in docs],
                        padding=True, truncation=True, max_length=512, return_tensors='pt')
        logits = ce_mod_ha(**enc).logits.squeeze(-1)
        if logits.dim() == 0: logits = logits.unsqueeze(0)
    return [d for _, d in sorted(zip(logits.tolist(), docs), key=lambda x: x[0], reverse=True)[:5]]

def retrieve_ha(q):
    b = bm25_ha.invoke(q)
    c = vdb_ha.similarity_search(q, k=10)
    sc, dm = {}, {}
    for rk, d in enumerate(b):
        k = d.page_content[:200]; sc[k] = sc.get(k,0.0) + 0.4/(60+rk+1); dm[k] = d
    for rk, d in enumerate(c):
        k = d.page_content[:200]; sc[k] = sc.get(k,0.0) + 0.6/(60+rk+1); dm[k] = d
    return rerank_ha(q, [dm[k] for k in sorted(sc, key=sc.__getitem__, reverse=True)[:10]])

print('Cargando Ollama para HyDE...')
llm_hyde_ha = OllamaLLM(model='llama3:8b', temperature=0.0)

print('Cargando generador fine-tuned 4-bit...')
tok_ha = AutoTokenizer.from_pretrained(MODEL_ID)
tok_ha.pad_token = tok_ha.eos_token
mod_ha = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_cfg, trust_remote_code=True)
mod_ha = PeftModel.from_pretrained(mod_ha, adapter_path)
mod_ha.eval()
print('Todo listo')

results_ha = []
print(f'Hibrido Avanzado ({N_SAMPLES} preguntas)...')
for i, e in enumerate(dataset):
    t0 = time.perf_counter()
    q_exp = llm_hyde_ha.invoke(
        'Genera un parrafo breve de documentacion legal del RGPD que responderia '
        'directamente a esta pregunta. Solo el fragmento, sin preambulo.\n\n'
        f'Pregunta: {e["question"]}\nFragmento hipotetico:'
    ).strip()
    docs = retrieve_ha(q_exp)
    ctx = '\n\n'.join(d.page_content for d in docs)
    prompt = (
        '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n'
        'Eres un asistente legal experto en el RGPD. Responde basandote UNICAMENTE en el contexto.'
        '<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n'
        f'Contexto: {ctx}\n\nPregunta: {e["question"]}'
        '<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
    )
    inputs = tok_ha(prompt, return_tensors='pt', truncation=True, max_length=4096).to('cuda')
    with torch.no_grad():
        out = mod_ha.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True,
                               repetition_penalty=1.2, eos_token_id=tok_ha.eos_token_id)
    inp_len = inputs['input_ids'].shape[1]
    resp = tok_ha.decode(out[0][inp_len:], skip_special_tokens=True).strip()
    lat = round(time.perf_counter() - t0, 3)
    results_ha.append({'question': e['question'], 'answer': resp, 'contexts': [d.page_content for d in docs], 'ground_truth': e['ground_truth'], 'latency_s': lat})
    print(f'  [{i+1}/{N_SAMPLES}] {lat}s')

lats = [r['latency_s'] for r in results_ha]
print(f'\nHibrido Avanzado => media: {sum(lats)/len(lats):.2f}s | min: {min(lats):.2f}s | max: {max(lats):.2f}s')
with open(f'{OUTPUT_DIR}/hybrid_advanced_latency_results.json', 'w', encoding='utf-8') as f:
    json.dump(results_ha, f, ensure_ascii=False, indent=4)
all_results['Hibrido Avanzado'] = lats

del mod_ha, tok_ha, vdb_ha, bm25_ha, ce_mod_ha, ce_tok_ha, emb_ha
gc.collect(); torch.cuda.empty_cache()

## Resumen — Tabla de latencias

In [ ]:
# CELDA 11 — Resumen + guardar CSV consolidado
import csv

print('=' * 60)
print(f'{"Arquitectura":<28} {"Media(s)":>9} {"Min(s)":>7} {"Max(s)":>7}')
print('-' * 60)

rows = []
for arch, lats in all_results.items():
    media = sum(lats) / len(lats)
    print(f'{arch:<28} {media:>9.2f} {min(lats):>7.2f} {max(lats):>7.2f}')
    rows.append({'arquitectura': arch, 'media_s': round(media, 3),
                 'min_s': round(min(lats), 3), 'max_s': round(max(lats), 3),
                 'n_samples': len(lats)})

print('=' * 60)

csv_path = f'{OUTPUT_DIR}/latency_summary.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['arquitectura', 'media_s', 'min_s', 'max_s', 'n_samples'])
    w.writeheader(); w.writerows(rows)

print(f'\nResumen guardado en: {csv_path}')
print(f'JSONs individuales en: {OUTPUT_DIR}/')